# NeoOLAF SURE Run From Layer 2, Parallel

This notebook uses the safest recoverable strategy:

1. **Do not resume from Layer 3.**
2. Find only valid `layer02_candidate_enrichment/state.json` files accepted by `PipelineState.load_json`.
3. Pick the latest valid Layer 2 state with non-empty enriched expressions.
4. Run **Layers 3 to 12** into a clean new run directory.
5. Use parallel workers.
6. Verify the final exports.

This avoids the broken partial Layer 3 state problem.

In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess
import shlex
import platform
from datetime import datetime

print("Python executable:", sys.executable)
print("Platform:", platform.platform())
print("Current working directory:", Path.cwd())

## 0. Configuration

This notebook is intentionally conservative about state selection and aggressive about parallelism.

You usually only need to check `PROJECT_ROOT` and `CHOSEN_LAYER2_INDEX`.

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Leave None for auto-detection.
PROJECT_ROOT = None

# If auto-detection fails, set it explicitly, for example:
# PROJECT_ROOT = Path(r"C:\Users\henri\Documents\git\post-doc\NeoOLAF")
# PROJECT_ROOT = Path("/home/galencarmedeiro/git/postdoc/NeoOLAF")

# Search only inside this runs folder when possible.
# Leave None to use PROJECT_ROOT / examples / XQualityMachine32 / runs / xquality_machine32.
XQUALITY_RUNS_ROOT = None

# Which valid Layer 2 state to use from the printed candidate list.
# 1 means latest/best candidate.
CHOSEN_LAYER2_INDEX = 1

# Model/profile/backend used for the resume.
MODEL = "openrouter/openai/gpt-oss-20b"
PROFILE = "xquality_loose"
RAG_BACKEND = "agentic"

# Parallelism. This is parallel, as requested.
WORKERS = 4

# Run target.
FROM_LAYER = 3
TO_LAYER = 12

# Output folder name. A timestamp is appended automatically.
OUTPUT_RUN_NAME_PREFIX = "sure_resume_from_layer02_parallel"

# Actually run the expensive command.
# Keep True. Set False only to dry-run.
RUN_NEOOLAF = True

# If True, the notebook refuses to run if the Layer 2 state has no enriched expressions.
ABORT_IF_LAYER2_LOOKS_EMPTY = True

## 1. Resolve project root and import NeoOLAF

This cell must pass before spending money.

In [ ]:
def looks_like_neoolaf_root(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    return (
        (path / "src" / "neoolaf").exists()
        or (path / "pyproject.toml").exists()
        or (path / "experiments").exists()
    )

def candidate_project_roots():
    cwd = Path.cwd()
    candidates = [cwd, *cwd.parents]

    for key in ["NEOOLAF_PROJECT_ROOT", "PROJECT_ROOT"]:
        val = os.environ.get(key)
        if val:
            candidates.append(Path(val).expanduser())

    candidates.extend([
        Path(r"C:\Users\henri\Documents\git\post-doc\NeoOLAF"),
        Path.home() / "Documents" / "git" / "post-doc" / "NeoOLAF",
        Path("/home/galencarmedeiro/git/postdoc/NeoOLAF"),
        Path.home() / "git" / "postdoc" / "NeoOLAF",
    ])

    out = []
    seen = set()
    for p in candidates:
        try:
            rp = p.resolve()
        except Exception:
            rp = p
        key = str(rp).lower()
        if key not in seen:
            seen.add(key)
            out.append(rp)
    return out

if PROJECT_ROOT is None:
    matches = [p for p in candidate_project_roots() if looks_like_neoolaf_root(p)]
    if not matches:
        print("Tried these paths:")
        for p in candidate_project_roots():
            print(" -", p)
        raise RuntimeError("Could not find NeoOLAF root. Set PROJECT_ROOT manually in configuration.")
    PROJECT_ROOT = matches[0]
else:
    PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
    if not looks_like_neoolaf_root(PROJECT_ROOT):
        raise RuntimeError(f"PROJECT_ROOT does not look like NeoOLAF root: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)

src = PROJECT_ROOT / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

from neoolaf.core.pipeline_state import PipelineState

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PWD:", Path.cwd())
print("Imported PipelineState OK.")

## 2. Find valid Layer 2 PipelineStates only

This cell ignores every invalid `state.json`. It only accepts files that NeoOLAF can load as a real `PipelineState`.

In [ ]:
LAYER2_REL = Path("layer02_candidate_enrichment") / "state.json"

if XQUALITY_RUNS_ROOT is None:
    XQUALITY_RUNS_ROOT = PROJECT_ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32"
else:
    XQUALITY_RUNS_ROOT = Path(XQUALITY_RUNS_ROOT).expanduser().resolve()

if not XQUALITY_RUNS_ROOT.exists():
    raise RuntimeError(f"XQUALITY_RUNS_ROOT does not exist: {XQUALITY_RUNS_ROOT}")

def safe_len(value):
    if value is None:
        return 0
    try:
        return len(value)
    except TypeError:
        return 0

def load_pipeline_state(path: Path):
    return PipelineState.load_json(str(path))

def state_count(state, name):
    value = getattr(state, name, None)
    return safe_len(value)

candidates = []

for p in XQUALITY_RUNS_ROOT.rglob(str(LAYER2_REL).replace("\\", "/")):
    try:
        st = load_pipeline_state(p)
    except Exception as e:
        continue

    enriched = state_count(st, "enriched_expressions")
    linguistic = state_count(st, "linguistic_expressions")
    entities = state_count(st, "entity_candidates")
    relations = state_count(st, "relation_candidates")

    candidates.append({
        "path": p,
        "run_dir": p.parent.parent,
        "mtime": p.stat().st_mtime,
        "state": st,
        "linguistic_expressions": linguistic,
        "enriched_expressions": enriched,
        "entity_candidates": entities,
        "relation_candidates": relations,
    })

# Prefer non-empty Layer 2 states, newest first.
candidates.sort(
    key=lambda x: (
        x["enriched_expressions"] > 0,
        x["linguistic_expressions"] > 0,
        x["mtime"],
    ),
    reverse=True,
)

if not candidates:
    raise RuntimeError(
        f"No valid Layer 2 PipelineState found under {XQUALITY_RUNS_ROOT}. "
        "This means you do not have a recoverable Layer 2 checkpoint there."
    )

print("VALID LAYER 2 CANDIDATES")
print("=" * 120)
for i, c in enumerate(candidates[:30], 1):
    print(f"{i:02d} | {datetime.fromtimestamp(c['mtime']).strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"     run_dir: {c['run_dir']}")
    print(f"     state:   {c['path']}")
    print(
        "     counts: "
        f"linguistic={c['linguistic_expressions']}, "
        f"enriched={c['enriched_expressions']}, "
        f"entities={c['entity_candidates']}, "
        f"relations={c['relation_candidates']}"
    )

## 3. Select the Layer 2 state

This is the only state used. The notebook never uses a Layer 3 partial state.

In [ ]:
if CHOSEN_LAYER2_INDEX < 1 or CHOSEN_LAYER2_INDEX > len(candidates):
    raise RuntimeError(f"CHOSEN_LAYER2_INDEX={CHOSEN_LAYER2_INDEX} is out of range.")

chosen = candidates[CHOSEN_LAYER2_INDEX - 1]
LAYER2_STATE = chosen["path"]
SOURCE_RUN_DIR = chosen["run_dir"]

if ABORT_IF_LAYER2_LOOKS_EMPTY and chosen["enriched_expressions"] <= 0:
    raise RuntimeError(
        "Chosen Layer 2 state has zero enriched expressions. "
        "Choose another candidate or rerun Layers 1-2 first."
    )

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_RUN_DIR = XQUALITY_RUNS_ROOT / f"{OUTPUT_RUN_NAME_PREFIX}_{timestamp}"
OUTPUT_RUN_DIR.mkdir(parents=True, exist_ok=True)

print("USING LAYER 2 STATE:")
print(LAYER2_STATE)
print()
print("SOURCE RUN DIR:")
print(SOURCE_RUN_DIR)
print()
print("NEW OUTPUT RUN DIR:")
print(OUTPUT_RUN_DIR)
print()
print("COUNTS:")
for k in ["linguistic_expressions", "enriched_expressions", "entity_candidates", "relation_candidates"]:
    print(k, "=", chosen[k])

## 4. Build the command

This command reruns only Layers 3 to 12, with parallel workers.

In [ ]:
cmd = [
    sys.executable, "-m", "neoolaf", "run",
    "--resume-from", str(LAYER2_STATE),
    "--run-dir", str(OUTPUT_RUN_DIR),
    "--from-layer", str(FROM_LAYER),
    "--to-layer", str(TO_LAYER),
    "--model", MODEL,
    "--profile", PROFILE,
    "--max-concurrency-layer03", str(WORKERS),
    "--max-concurrency-layer04", str(WORKERS),
    "--max-concurrency-layer05", str(WORKERS),
    "--max-concurrency-layer06", str(WORKERS),
    "--max-concurrency-layer07", str(WORKERS),
    "--max-concurrency-layer08", str(WORKERS),
    "--max-concurrency-layer09", str(WORKERS),
    "--max-concurrency-layer10", str(WORKERS),
    "--max-concurrency-layer11", str(WORKERS),
    "--max-concurrency-layer12", str(WORKERS),
    "--verbose",
]

if RAG_BACKEND:
    cmd.extend(["--rag-backend", RAG_BACKEND])

print("COMMAND")
print("=" * 120)
print(" ".join(shlex.quote(str(x)) for x in cmd))

## 5. Run NeoOLAF

This runs the command directly, with live output.

If this fails, the failure is from the actual NeoOLAF run/API, not from a wrong state file.

In [ ]:
if not RUN_NEOOLAF:
    print("RUN_NEOOLAF=False. Dry run only.")
else:
    print("Starting NeoOLAF at", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print("Output run dir:", OUTPUT_RUN_DIR)

    env = os.environ.copy()
    result = subprocess.run(
        cmd,
        cwd=PROJECT_ROOT,
        env=env,
        check=False,
    )

    print("Finished at", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print("Return code:", result.returncode)

    if result.returncode != 0:
        raise RuntimeError(
            f"NeoOLAF failed with return code {result.returncode}. "
            f"Partial outputs, if any, are in: {OUTPUT_RUN_DIR}"
        )

## 6. Verify final exports

This confirms whether Layer 12 produced the expected KG and ontology files.

In [ ]:
exports = OUTPUT_RUN_DIR / "exports"
layer12_state = OUTPUT_RUN_DIR / "layer12_serialization" / "state.json"
checkpoint = OUTPUT_RUN_DIR / "checkpoints" / "after_selected_pipeline.pkl.gz"

print("OUTPUT_RUN_DIR:", OUTPUT_RUN_DIR)
print("exports exists:", exports.exists(), exports)
print("layer12 state exists:", layer12_state.exists(), layer12_state)
print("final checkpoint exists:", checkpoint.exists(), checkpoint)

if not exports.exists():
    raise RuntimeError("Exports folder does not exist. Layer 12 probably did not finish.")

print("\nEXPORT FILES")
print("=" * 120)
for p in sorted(exports.iterdir()):
    print(p.name)

def count_triples(path: Path):
    if not path.exists():
        return None
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        return len(data.get("triples", []))
    except Exception as e:
        return f"ERROR: {e}"

for name in ["kg_local.json", "kg_inferred.json"]:
    p = exports / name
    print(f"\n{name}: exists={p.exists()} triples={count_triples(p)}")

for name in ["ontology_local.ttl", "ontology_inferred.ttl"]:
    p = exports / name
    print(f"{name}: exists={p.exists()} size={p.stat().st_size if p.exists() else 0} bytes")

## 7. Print final paths to copy

Use these paths in later evaluation cells.

In [ ]:
print("FINAL RUN DIR:")
print(OUTPUT_RUN_DIR)

print("\nEXPORTS:")
print(exports)

print("\nKG LOCAL:")
print(exports / "kg_local.json")

print("\nKG INFERRED:")
print(exports / "kg_inferred.json")

print("\nONTOLOGY LOCAL:")
print(exports / "ontology_local.ttl")

print("\nONTOLOGY INFERRED:")
print(exports / "ontology_inferred.ttl")

print("\nLAYER 12 STATE:")
print(layer12_state)